# 00 · Run the M5 pipeline end-to-end

Thin demo notebook that drives `src/m5` rather than implementing logic inline.
Heavy lifting lives in the package; this notebook is just a runnable script.

Prereqs: `make bootstrap && make prep`.

In [ ]:
from pathlib import Path
import pandas as pd
from m5.config import SETTINGS, set_global_seed
from m5.cv import stats_cv, lgbm_cv
from m5.evaluation import compute_components, wrmsse_for_models

set_global_seed()
long_path = SETTINGS.processed_dir / 'long.parquet'
df = pd.read_parquet(long_path)
df.head()

## Statistical baselines (Theta + AutoETS + SeasonalNaive)

In [ ]:
cv_stats = stats_cv(df, h=SETTINGS.horizon, n_windows=SETTINGS.n_windows)
components = compute_components(df[df['ds'] < cv_stats['ds'].min()])
stats_scores = wrmsse_for_models(cv_stats[['unique_id','ds','y']], cv_stats, components)
stats_scores

## LightGBM global model

In [ ]:
cv_lgbm = lgbm_cv(df, h=SETTINGS.horizon, n_windows=SETTINGS.n_windows)
lgbm_scores = wrmsse_for_models(cv_lgbm[['unique_id','ds','y']], cv_lgbm, components)
lgbm_scores

## Combined leaderboard

In [ ]:
leaderboard = pd.concat([stats_scores, lgbm_scores]).sort_values()
leaderboard.to_frame('WRMSSE')